# Arquitectura backend + base de experimentación (issue #12)
Usa los **mismos casos de uso** que la API (`backend/src/recoleccion/`).
Para cambiar dataset / modelo / configuración alcanza con editar la celda de parámetros: sin copiar lógica entre celdas.

In [ ]:
import sys
from pathlib import Path

# Funciona desde la raíz del repo y desde notebooks/ (nbval ejecuta
# el kernel con cwd = directorio del notebook).
CANDIDATOS = [Path.cwd() / "backend" / "src",
              Path.cwd().parent / "backend" / "src"]
SRC = next(c for c in CANDIDATOS if (c / "recoleccion").is_dir())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from recoleccion import composicion
print("src:", SRC.exists())

## Parámetros (el único lugar que hay que editar)

In [ ]:
# Cambiar dataset/modelo/configuración SOLO acá.
# SRC viene de la celda de setup; el default apunta al fixture local.
DATASET_DIR = str(SRC.parent / "tests" / "fixtures")  # nodos.csv + tramos.csv (esquema v1)
MODELO_ID = "arbol"  # arbol | logistica | regla-v1
INCLUIR = ("media", "alta")  # criterio de selección de tramos
SEMILLA = 0

## 1. Validar / cargar dataset

In [ ]:
nodos, tramos = composicion.cargar_dataset_v1(DATASET_DIR)
print(f"nodos={len(nodos)} tramos={len(tramos)}")
assert len(tramos) > 0, "dataset sin tramos"

## 2. Comparar modelos (offline, protocolo reproducible)

In [ ]:
from recoleccion.dominio import etiqueta_v1

etiquetas = {t.tramo_id: etiqueta_v1(t.dias_desde_ultima_recoleccion,
    t.historico_quejas_vecinales, t.cantidad_contenedores, t.tipo_zona)
    for t in tramos}
filas = composicion.filas_para_ml(tramos, etiquetas)
comp = composicion.comparar_demos(filas, semilla=SEMILLA)
for m in comp.metricas:
    print(f"{m.modelo_id}: acc={m.accuracy:.2f} f1={m.f1_macro:.2f} (train={m.n_train} test={m.n_test})")
print("baseline:", round(comp.baseline_accuracy, 2), "| mejor_por_cv:", comp.mejor_por_cv)
assert comp.mejor_por_cv in ("arbol", "logistica")
assert comp.registro["semilla"] == SEMILLA

## 3. Predecir prioridades (inferencia, sin entrenar acá)

In [ ]:
preds = composicion.predecir_con(MODELO_ID, filas, tramos, semilla=SEMILLA)
print([(p.tramo_id, p.prioridad) for p in preds[:4]], "...")
assert [p.tramo_id for p in preds] == [t.tramo_id for t in tramos], "asociación tramo<->predicción rota"
assert {p.prioridad for p in preds} <= {"baja", "media", "alta"}

## 4. Planificar recolección (Hierholzer / cartero chino)

In [ ]:
rec = composicion.planificar(nodos, tramos, preds, incluir=INCLUIR)
print(f"pasos={len(rec.orden)} repeticiones={list(rec.repeticiones)} costo={rec.costo_total_m:.1f} {rec.unidad}")
pri = {p.tramo_id: p.prioridad for p in preds}
esperados = {t.tramo_id for t in tramos if pri[t.tramo_id] in INCLUIR}
assert esperados <= set(rec.orden), "el recorrido no cubre los tramos seleccionados"
assert rec.costo_total_m > 0

> **Aviso:** la etiqueta es *construida* con la regla v1: el modelo aprende esa regla y una buena métrica **no** demuestra utilidad operativa real.